In [ ]:
# ============================================================================
# CaptainCook4D Feature Extraction Notebook
# Step 2 Part 2: Feature Extraction with EgoVLP/PerceptionEncoder
# ============================================================================
# This notebook extracts video features using:
# 1. EgoVLP - Video-Language Pretraining model (NeurIPS 2022)
# 2. PerceptionEncoder - Alternative video-text model
#
# Features are saved in .npz format compatible with the baseline code.
# ============================================================================

import os
import sys

# Clone feature extractors repository (optional, for Omnivore/SlowFast)
if not os.path.exists("extractors"):
    !git clone --recursive https://github.com/CaptainCook4D/feature_extractors.git extractors

Cloning into 'extractors'...
remote: Enumerating objects: 858, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 858 (delta 0), reused 1 (delta 0), pack-reused 855 (from 1)
Receiving objects: 100% (858/858), 115.02 KiB | 965.00 KiB/s, done.
Resolving deltas: 100% (603/603), done.
Submodule 'lib/imagebind' (https://github.com/rohithpeddi/ImageBind.git) registered for path 'lib/imagebind'
Cloning into '/content/extractors/lib/imagebind'...
remote: Enumerating objects: 139, done.        
remote: Counting objects: 100% (87/87), done.        
remote: Compressing objects: 100% (45/45), done.        
remote: Total 139 (delta 64), reused 42 (delta 42), pack-reused 52 (from 1)        
Receiving objects: 100% (139/139), 2.64 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (67/67), done.
Submodule path 'lib/imagebind': checked out 'ed9a64eec34bcc507c7ab5793b461aef537cf77b'


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU availability
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU required. Go to 'Runtime > Change runtime type' and select 'T4 GPU'.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU: NVIDIA L4


In [ ]:
# Install dependencies
!pip install -q transformers timm ftfy einops decord pytorchvideo natsort

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 61.6 MB/s eta 0:00:00


## 1. Data Preparation
Extract the resized CaptainCook4D videos from Google Drive.

In [ ]:
# Configuration
DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
VIDEOS_ZIP = f"{DRIVE_BASE_PATH}/captain_cook_4d_gopro_resized.zip"
VIDEO_DIR = "data/videos"
FEATURE_DIR = "data/features"

import os

# 1. Copy the zip from Drive to the local Colab instance (One big fast read)
if os.path.exists(VIDEOS_ZIP):
    print("Copying zip to local disk...")
    !cp "{VIDEOS_ZIP}" /content/temp_videos.zip

    # 2. Unzip from the local file (Extremely fast)
    print("Unzipping locally...")
    !mkdir -p {VIDEO_DIR}
    !unzip -q -o /content/temp_videos.zip -d {VIDEO_DIR}/

    # 3. Remove the local zip to free up RAM/Disk space
    !rm /content/temp_videos.zip
else:
    print(f"Warning: Zip file not found at {VIDEOS_ZIP}")

# List available videos
video_files = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])
print(f"Found {len(video_files)} video files")
print(f"Sample: {video_files[:5]}")

Copying zip to local disk...
Unzipping locally...
Found 384 video files
Sample: ['10_16_360p_224.mp4', '10_18_360p_224.mp4', '10_24_360p_224.mp4', '10_26_360p_224.mp4', '10_31_360p_224.mp4']


## 2. EgoVLP Feature Extraction

EgoVLP (Ego-centric Video-Language Pretraining) is a model from NeurIPS 2022 that learns aligned video-text embeddings from egocentric video data (Ego4D). We use its video encoder to extract features.

**Key Parameters:**
- Input resolution: 224x224
- Number of frames: 4 (default, adjustable)
- Feature dimension: 256 (after projection)
- Architecture: SpaceTimeTransformer (TimeSformer-based)

### 2.1 Download EgoVLP Checkpoint

In [ ]:
import os
import gdown

# Create checkpoint directory
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Download EgoVLP pretrained checkpoint
EGOVLP_CKPT = f"{CKPT_DIR}/egovlp.pth"
if not os.path.exists(EGOVLP_CKPT):
    print("Downloading EgoVLP checkpoint...")
    # EgoVLP_PT_BEST from Google Drive
    gdown.download(
        "https://drive.google.com/uc?id=1-cP3Gcg0NGDcMZalgJ_615BQdbFIbcj7",
        EGOVLP_CKPT,
        quiet=False
    )
    print(f"Checkpoint saved to {EGOVLP_CKPT}")
else:
    print(f"Checkpoint already exists: {EGOVLP_CKPT}")

# Download ViT base weights (required by EgoVLP)
VIT_CKPT = f"{CKPT_DIR}/jx_vit_base_p16_224-80ecf9dd.pth"
if not os.path.exists(VIT_CKPT):
    print("Downloading ViT base checkpoint...")
    !wget -q -O {VIT_CKPT} https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
    print(f"ViT checkpoint saved to {VIT_CKPT}")
else:
    print(f"ViT checkpoint already exists: {VIT_CKPT}")

Downloading...
From (original): https://drive.google.com/uc?id=1-cP3Gcg0NGDcMZalgJ_615BQdbFIbcj7
From (redirected): https://drive.google.com/uc?id=1-cP3Gcg0NGDcMZalgJ_615BQdbFIbcj7&confirm=t&uuid=33b0796d-dcb2-4032-bab1-6a4b22b3d391
To: /content/checkpoints/egovlp.pth
100%|██████████| 2.17G/2.17G [00:22<00:00, 95.9MB/s]


Checkpoint saved to checkpoints/egovlp.pth
ViT checkpoint saved to checkpoints/jx_vit_base_p16_224-80ecf9dd.pth


### 2.2 Define EgoVLP Model Architecture

We implement the EgoVLP model components directly to avoid cloning the full repository.

In [ ]:
"""
SpaceTimeTransformer implementation for EgoVLP
Based on: https://github.com/showlab/EgoVLP/blob/main/model/video_transformer.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from collections import OrderedDict
from einops import rearrange, repeat
from timm.models.layers import DropPath, to_2tuple, trunc_normal_


def attn(q, k, v):
    """Attention operation."""
    sim = torch.einsum('b i d, b j d -> b i j', q, k)
    attn_weights = sim.softmax(dim=-1)
    out = torch.einsum('b i j, b j d -> b i d', attn_weights, v)
    return out


class Mlp(nn.Module):
    """MLP block."""
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class VideoPatchEmbed(nn.Module):
    """Video to Patch Embedding."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, num_frames=8):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0]) * num_frames
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches
        self.num_frames = num_frames
        self.embed_dim = embed_dim
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, F, C, H, W = x.shape
        assert F <= self.num_frames
        x = x.view(-1, C, H, W)
        x = self.proj(x)
        return x


class VarAttention(nn.Module):
    """Variable attention for space-time transformer."""
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None,
                 attn_drop=0., proj_drop=0., initialize='random'):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

        if initialize == 'zeros':
            self.qkv.weight.data.fill_(0)
            self.qkv.bias.data.fill_(0)
            self.proj.weight.data.fill_(1)
            self.proj.bias.data.fill_(0)

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, einops_from, einops_to, **einops_dims):
        h = self.num_heads
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        q = q * self.scale

        # Splice out CLS token
        (cls_q, q_), (cls_k, k_), (cls_v, v_) = map(lambda t: (t[:, 0:1], t[:, 1:]), (q, k, v))

        # CLS token attends to all patches
        cls_out = attn(cls_q, k, v)

        # Rearrange for space/time attention
        q_, k_, v_ = map(lambda t: rearrange(t, f'{einops_from} -> {einops_to}', **einops_dims), (q_, k_, v_))

        # Expand CLS token
        r = q_.shape[0] // cls_k.shape[0]
        cls_k, cls_v = map(lambda t: repeat(t, 'b () d -> (b r) () d', r=r), (cls_k, cls_v))
        k_ = torch.cat((cls_k, k_), dim=1)
        v_ = torch.cat((cls_v, v_), dim=1)

        # Attention
        out = attn(q_, k_, v_)
        out = rearrange(out, f'{einops_to} -> {einops_from}', **einops_dims)
        out = torch.cat((cls_out, out), dim=1)
        out = rearrange(out, '(b h) n d -> b n (h d)', h=h)

        x = self.proj(out)
        x = self.proj_drop(x)
        return x


class SpaceTimeBlock(nn.Module):
    """Space-Time Transformer Block."""
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None,
                 drop=0., attn_drop=0., drop_path=0., act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm, time_init='zeros', attention_style='frozen-in-time'):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = VarAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                  qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.timeattn = VarAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                      qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop,
                                      initialize=time_init)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.norm3 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        self.attention_style = attention_style

    def forward(self, x, einops_from_space, einops_to_space, einops_from_time, einops_to_time,
                time_n, space_f):
        time_output = self.timeattn(self.norm3(x), einops_from_time, einops_to_time, n=time_n)
        time_residual = x + time_output
        space_output = self.attn(self.norm1(time_residual), einops_from_space, einops_to_space, f=space_f)
        space_residual = x + self.drop_path(space_output)
        x = space_residual + self.drop_path(self.mlp(self.norm2(space_residual)))
        return x


class SpaceTimeTransformer(nn.Module):
    """Space-Time Transformer for video understanding."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., qkv_bias=True,
                 qk_scale=None, representation_size=None, drop_rate=0., attn_drop_rate=0.,
                 drop_path_rate=0., norm_layer=None, num_frames=8, time_init='rand',
                 attention_style='frozen-in-time'):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.num_frames = num_frames
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)

        self.patch_embed = VideoPatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans,
            embed_dim=embed_dim, num_frames=num_frames)
        num_patches = self.patch_embed.num_patches
        self.patches_per_frame = num_patches // num_frames

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patches_per_frame + 1, embed_dim))
        self.temporal_embed = nn.Parameter(torch.zeros(1, num_frames, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            SpaceTimeBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                          qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate,
                          attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                          time_init=time_init, attention_style=attention_style)
            for i in range(depth)])
        self.norm = norm_layer(embed_dim)

        if representation_size:
            self.num_features = representation_size
            self.pre_logits = nn.Sequential(OrderedDict([
                ('fc', nn.Linear(embed_dim, representation_size)),
                ('act', nn.Tanh())
            ]))
        else:
            self.pre_logits = nn.Identity()

        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()

        trunc_normal_(self.pos_embed, std=.02)
        trunc_normal_(self.cls_token, std=.02)

        if num_frames == 1:
            self.apply(self._init_weights)

        # Einops patterns
        self.einops_from_space = 'b (f n) d'
        self.einops_to_space = '(b f) n d'
        self.einops_from_time = 'b (f n) d'
        self.einops_to_time = '(b n) f d'

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        b, curr_frames, channels, _, _ = x.shape
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(2, 1)
        x = x.reshape(b, -1, self.patch_embed.embed_dim)

        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Positional embedding
        cls_embed = self.pos_embed[:, 0, :].unsqueeze(1)
        tile_pos_embed = self.pos_embed[:, 1:, :].repeat(1, self.num_frames, 1)
        tile_temporal_embed = self.temporal_embed.repeat_interleave(self.patches_per_frame, 1)
        total_pos_embed = tile_pos_embed + tile_temporal_embed
        total_pos_embed = torch.cat([cls_embed, total_pos_embed], dim=1)

        curr_patches = x.shape[1]
        x = x + total_pos_embed[:, :curr_patches]
        x = self.pos_drop(x)

        n = self.patches_per_frame
        f = curr_frames

        for blk in self.blocks:
            x = blk(x, self.einops_from_space, self.einops_to_space,
                   self.einops_from_time, self.einops_to_time, time_n=n, space_f=f)

        x = self.norm(x)[:, 0]
        x = self.pre_logits(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

print("SpaceTimeTransformer defined successfully.")

SpaceTimeTransformer defined successfully.


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
"""
EgoVLP Model (FrozenInTime architecture)
Based on: https://github.com/showlab/EgoVLP/blob/main/model/model.py
"""

import torch
import torch.nn as nn
from transformers import AutoModel
import sys
from types import ModuleType

# Mock parse_config module to fix checkpoint loading
if 'parse_config' not in sys.modules:
    mock_parse_config = ModuleType('parse_config')
    class ConfigParser:
        pass
    mock_parse_config.ConfigParser = ConfigParser
    sys.modules['parse_config'] = mock_parse_config

def state_dict_data_parallel_fix(load_state_dict, curr_state_dict):
    """Fix state dict keys from DataParallel training."""
    new_state_dict = {}
    for k, v in load_state_dict.items():
        # Remove 'module.' prefix if present
        name = k.replace('module.', '') if k.startswith('module.') else k
        new_state_dict[name] = v
    return new_state_dict


class FrozenInTime(nn.Module):
    """
    EgoVLP Model: FrozenInTime architecture with video and text encoders.
    For feature extraction, we only use the video encoder.
    """
    def __init__(self, video_params, text_params, projection_dim=256,
                 load_checkpoint=None, projection='minimal', load_temporal_fix='zeros'):
        super().__init__()
        self.video_params = video_params
        self.text_params = text_params
        self.load_temporal_fix = load_temporal_fix

        # Text model (optional, for text features)
        if text_params.get('pretrained', True):
            model_name = text_params.get('model', 'distilbert-base-uncased')
            if model_name.startswith('distilbert'):
                self.text_model = AutoModel.from_pretrained('distilbert-base-uncased')
            else:
                self.text_model = AutoModel.from_pretrained(model_name)
        else:
            self.text_model = None

        # Video model
        if video_params['model'] == "SpaceTimeTransformer":
            num_frames = video_params.get('num_frames', 4)
            time_init = video_params.get('time_init', 'zeros')
            attention_style = video_params.get('attention_style', 'frozen-in-time')

            # Load ViT weights
            vit_model = torch.load(VIT_CKPT, map_location="cpu", weights_only=False)

            model = SpaceTimeTransformer(
                num_frames=num_frames,
                time_init=time_init,
                attention_style=attention_style
            )
            model.head = nn.Identity()
            model.pre_logits = nn.Identity()
            ftr_dim = model.embed_dim

            # Load ViT weights into SpaceTimeTransformer
            if load_checkpoint in ["", None]:
                new_vit_dict = state_dict_data_parallel_fix(vit_model, model.state_dict())
                model.load_state_dict(new_vit_dict, strict=False)

            self.video_model = model
        else:
            raise NotImplementedError(f"{video_params['model']} not implemented")

        self.video_model.fc = nn.Identity()

        # Projection layers
        if projection == 'minimal':
            if self.text_model is not None:
                self.txt_proj = nn.Sequential(
                    nn.ReLU(),
                    nn.Linear(self.text_model.config.hidden_size, projection_dim)
                )
            self.vid_proj = nn.Sequential(nn.Linear(ftr_dim, projection_dim))
        else:
            self.txt_proj = nn.Identity()
            self.vid_proj = nn.Identity()

        # Load full EgoVLP checkpoint
        if load_checkpoint not in ["", None]:
            checkpoint = torch.load(load_checkpoint, map_location='cpu', weights_only=False)
            state_dict = checkpoint['state_dict']
            new_state_dict = state_dict_data_parallel_fix(state_dict, self.state_dict())
            new_state_dict = self._inflate_positional_embeds(new_state_dict)
            self.load_state_dict(new_state_dict, strict=False)
            print(f"Loaded EgoVLP checkpoint from {load_checkpoint}")

    def _inflate_positional_embeds(self, new_state_dict):
        """Handle temporal embedding size mismatch."""
        curr_keys = list(self.state_dict().keys())
        if 'video_model.temporal_embed' in new_state_dict and 'video_model.temporal_embed' in curr_keys:
            load_temporal_embed = new_state_dict['video_model.temporal_embed']
            load_num_frames = load_temporal_embed.shape[1]
            curr_num_frames = self.video_params['num_frames']
            embed_dim = load_temporal_embed.shape[2]

            if load_num_frames != curr_num_frames:
                if load_num_frames > curr_num_frames:
                    new_temporal_embed = load_temporal_embed[:, :curr_num_frames, :]
                else:
                    if self.load_temporal_fix == 'zeros':
                        new_temporal_embed = torch.zeros([load_temporal_embed.shape[0], curr_num_frames, embed_dim])
                        new_temporal_embed[:, :load_num_frames] = load_temporal_embed
                    elif self.load_temporal_fix in ['interp', 'bilinear']:
                        mode = 'bilinear' if self.load_temporal_fix == 'bilinear' else 'nearest'
                        load_temporal_embed = load_temporal_embed.unsqueeze(0)
                        new_temporal_embed = F.interpolate(
                            load_temporal_embed, (curr_num_frames, embed_dim),
                            mode=mode, align_corners=True if mode == 'bilinear' else None
                        ).squeeze(0)
                    else:
                        raise NotImplementedError
                new_state_dict['video_model.temporal_embed'] = new_temporal_embed
        return new_state_dict

    def compute_video(self, video_data):
        """Extract video features."""
        video_embeddings = self.video_model(video_data)
        video_embeddings = self.vid_proj(video_embeddings)
        return video_embeddings

    def compute_text(self, text_data):
        """Extract text features."""
        if self.text_model is None:
            raise ValueError("Text model not initialized")
        text_embeddings = self.text_model(**text_data).last_hidden_state[:, 0, :]
        text_embeddings = self.txt_proj(text_embeddings)
        return text_embeddings

    def forward(self, data, video_only=False):
        """Forward pass."""
        if video_only:
            return self.compute_video(data['video'])
        text_embeddings = self.compute_text(data['text'])
        video_embeddings = self.compute_video(data['video'])
        return text_embeddings, video_embeddings

print("FrozenInTime (EgoVLP) model defined successfully.")

FrozenInTime (EgoVLP) model defined successfully.


### 2.3 Video Preprocessing

Define video transforms following EgoVLP specifications:
- Input resolution: 224x224
- Normalization: ImageNet mean/std
- Frame sampling: Uniform temporal subsampling

In [ ]:
import cv2
import numpy as np
import torch
from torchvision import transforms
try:
    from decord import VideoReader, cpu
    DECORD_AVAILABLE = True
except ImportError:
    DECORD_AVAILABLE = False
    print("Decord not installed, falling back to OpenCV")

def sample_frames_uniform(video_path, num_frames, segment_start=None, segment_end=None):
    """
    Sample frames uniformly from a video segment using Decord (fast) or OpenCV (fallback).
    """
    if DECORD_AVAILABLE:
        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            fps = vr.get_avg_fps()
            total_frames = len(vr)
            duration = total_frames / fps

            start_frame = int(segment_start * fps) if segment_start is not None else 0
            end_frame = int(segment_end * fps) if segment_end is not None else total_frames
            end_frame = min(end_frame, total_frames)

            if end_frame <= start_frame:
                # Handle edge case where segment is invalid
                return [np.zeros((224, 224, 3), dtype=np.uint8)] * num_frames

            frame_indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)
            frames = vr.get_batch(frame_indices).asnumpy()
            return [frames[i] for i in range(len(frames))]
        except Exception as e:
            print(f"Decord failed for {video_path}: {e}, falling back to OpenCV")

    # OpenCV Fallback
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start_frame = int(segment_start * fps) if segment_start is not None else 0
    end_frame = int(segment_end * fps) if segment_end is not None else total_frames
    end_frame = min(end_frame, total_frames)

    frame_indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        else:
            if frames:
                frames.append(frames[-1])
            else:
                frames.append(np.zeros((224, 224, 3), dtype=np.uint8))

    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8))

    return frames[:num_frames]


def get_video_transform(input_res=224):
    """Get video transform for EgoVLP."""
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(input_res),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


def preprocess_video_segment(video_path, num_frames=4, segment_start=None, segment_end=None):
    """
    Preprocess a video segment for EgoVLP.
    """
    frames = sample_frames_uniform(video_path, num_frames, segment_start, segment_end)
    transform = get_video_transform()

    transformed_frames = [transform(frame) for frame in frames]
    video_tensor = torch.stack(transformed_frames, dim=0).unsqueeze(0)

    return video_tensor


# Test preprocessing on first video
if video_files:
    test_video = os.path.join(VIDEO_DIR, video_files[0])
    test_tensor = preprocess_video_segment(test_video, num_frames=4)
    print(f"Preprocessed video shape: {test_tensor.shape}")
    print(f"Expected shape: (1, 4, 3, 224, 224)")

Preprocessed video shape: torch.Size([1, 4, 3, 224, 224])
Expected shape: (1, 4, 3, 224, 224)


### 2.4 Load EgoVLP Model

In [ ]:
# EgoVLP Configuration
NUM_FRAMES = 4  # Number of frames per segment
SEGMENT_LENGTH = 2  # Seconds per segment (matching Omnivore/SlowFast)
PROJECTION_DIM = 256  # Output feature dimension

video_params = {
    'model': 'SpaceTimeTransformer',
    'arch_config': 'base_patch16_224',
    'num_frames': NUM_FRAMES,
    'pretrained': True,
    'time_init': 'zeros',
    'attention_style': 'frozen-in-time',
}

text_params = {
    'model': 'distilbert-base-uncased',
    'pretrained': True,
    'input': 'text'
}

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading EgoVLP model...")
model = FrozenInTime(
    video_params=video_params,
    text_params=text_params,
    projection_dim=PROJECTION_DIM,
    load_checkpoint=EGOVLP_CKPT,
    projection='minimal',
    load_temporal_fix='bilinear'
)
model = model.to(device)
model.eval()

print(f"Model loaded on {device}")
print(f"Output feature dimension: {PROJECTION_DIM}")

Loading EgoVLP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loaded EgoVLP checkpoint from checkpoints/egovlp.pth
Model loaded on cuda
Output feature dimension: 256


### 2.5 Extract EgoVLP Features

Extract features for all videos using sliding window approach (2-second segments).

In [ ]:
from tqdm import tqdm
import math

def get_video_duration(video_path):
    """Get video duration in seconds using Decord or OpenCV."""
    if DECORD_AVAILABLE:
        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            return len(vr) / vr.get_avg_fps()
        except:
            pass

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total_frames / fps if fps > 0 else 0


def extract_egovlp_features(video_path, model, device, num_frames=4, segment_length=2, batch_size=32):
    """
    Extract EgoVLP features for a video using sliding window with BATCH PROCESSING.
    """
    duration = get_video_duration(video_path)
    if duration == 0:
        print(f"Warning: Could not read video {video_path}")
        return None

    num_segments = int(duration // segment_length)
    if num_segments == 0:
        return None

    # 1. Prepare all segments
    all_segments = []
    transform = get_video_transform()

    # Use Decord for fast batch reading if available
    if DECORD_AVAILABLE:
        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            fps = vr.get_avg_fps()
            total_frames = len(vr)

            for i in range(num_segments):
                start_time = i * segment_length
                end_time = start_time + segment_length

                start_frame = int(start_time * fps)
                end_frame = min(int(end_time * fps), total_frames)

                # Sample indices
                indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)

                # Get frames (Decord handles this efficiently)
                frames = vr.get_batch(indices).asnumpy()

                # Transform
                transformed = torch.stack([transform(f) for f in frames])
                all_segments.append(transformed)

        except Exception as e:
            print(f"Decord batch processing failed: {e}, falling back to slow loop")
            # Fallback logic would go here, but for now we assume Decord works or we use the loop below
            pass

    # Fallback or if Decord failed (and list is empty)
    if not all_segments:
        for i in range(num_segments):
            start_time = i * segment_length
            end_time = start_time + segment_length
            video_tensor = preprocess_video_segment(
                video_path, num_frames=num_frames,
                segment_start=start_time, segment_end=end_time
            )
            all_segments.append(video_tensor.squeeze(0))

    if not all_segments:
        return None

    # 2. Stack into a single tensor: (Num_Segments, Num_Frames, C, H, W)
    all_segments_tensor = torch.stack(all_segments)

    # 3. Process in batches
    features = []
    num_batches = math.ceil(len(all_segments_tensor) / batch_size)

    with torch.no_grad():
        for i in range(num_batches):
            batch = all_segments_tensor[i*batch_size : (i+1)*batch_size]
            batch = batch.to(device)

            # Model expects (B, F, C, H, W)
            batch_features = model.compute_video(batch)
            features.append(batch_features.cpu().numpy())

    return np.vstack(features)


# Test on a single video
if video_files:
    test_video = os.path.join(VIDEO_DIR, video_files[0])
    print(f"Testing optimized feature extraction on: {video_files[0]}")

    test_features = extract_egovlp_features(
        test_video, model, device,
        num_frames=NUM_FRAMES, segment_length=SEGMENT_LENGTH,
        batch_size=32
    )

    if test_features is not None:
        print(f"Extracted features shape: {test_features.shape}")
        duration = get_video_duration(test_video)
        print(f"Video duration: {duration:.1f}s, Segments: {int(duration // SEGMENT_LENGTH)}")

Testing optimized feature extraction on: 10_16_360p_224.mp4
Extracted features shape: (486, 256)
Video duration: 973.6s, Segments: 486


In [11]:
import torch
import math
import gc
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
try:
    from decord import VideoReader, cpu
    DECORD_AVAILABLE = True
except ImportError:
    DECORD_AVAILABLE = False

# Extract features for all videos
EGOVLP_FEATURE_DIR = f"{FEATURE_DIR}/egovlp"
os.makedirs(EGOVLP_FEATURE_DIR, exist_ok=True)

print(f"Extracting EgoVLP features for {len(video_files)} videos...")
print(f"Output directory: {EGOVLP_FEATURE_DIR}")

# --- 1. Define the Dataset (The "Worker" Logic) ---
class PreloadingVideoDataset(Dataset):
    def __init__(self, video_files, video_dir, num_frames, segment_length, transform):
        self.video_files = video_files
        self.video_dir = video_dir
        self.num_frames = num_frames
        self.segment_length = segment_length
        self.transform = transform

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        # This code now runs in PARALLEL on CPU workers!
        video_file = self.video_files[idx]
        video_path = os.path.join(self.video_dir, video_file)

        try:
            if DECORD_AVAILABLE:
                # Initialize Decord
                vr = VideoReader(video_path, ctx=cpu(0))
                duration = len(vr) / vr.get_avg_fps()

                num_segments = int(duration // self.segment_length)
                if num_segments == 0:
                    return torch.zeros(1), video_file, False # Valid but empty

                # Batch read ALL indices at once for speed
                fps = vr.get_avg_fps()
                total_frames = len(vr)

                # Pre-calculate all frame indices for all segments
                # Shape: (Num_Segments, Num_Frames)
                all_indices = []
                for i in range(num_segments):
                    start_time = i * self.segment_length
                    end_time = start_time + self.segment_length
                    start_frame = int(start_time * fps)
                    end_frame = min(int(end_time * fps), total_frames)
                    indices = np.linspace(start_frame, end_frame - 1, self.num_frames, dtype=int)
                    all_indices.append(indices)

                all_indices_flat = np.concatenate(all_indices)

                # The Heavy Lifting: Read & Transform
                # We read all needed frames in one massive batch
                frames = vr.get_batch(all_indices_flat).asnumpy()

                # Apply transform (This is slow, so we want workers to do it!)
                # Reshape to (Seg * Frame, H, W, C) -> Process -> Reshape back
                tensor_list = [self.transform(f) for f in frames]
                full_tensor = torch.stack(tensor_list)

                # Reshape to (Num_Segments, Num_Frames, C, H, W)
                C, H, W = full_tensor.shape[1:]
                full_tensor = full_tensor.view(num_segments, self.num_frames, C, H, W)

                return full_tensor, video_file, True
            else:
                # Fallback to OpenCV (slower, but works)
                # Note: This runs in the worker, so it's still parallelized!
                duration = get_video_duration(video_path)
                num_segments = int(duration // self.segment_length)
                if num_segments == 0:
                    return torch.zeros(1), video_file, False

                all_segments = []
                for i in range(num_segments):
                    start_time = i * self.segment_length
                    end_time = start_time + self.segment_length
                    video_tensor = preprocess_video_segment(
                        video_path, num_frames=self.num_frames,
                        segment_start=start_time, segment_end=end_time
                    )
                    all_segments.append(video_tensor.squeeze(0))

                full_tensor = torch.stack(all_segments)
                return full_tensor, video_file, True

        except Exception as e:
            # print(f"Error loading {video_file}: {e}") # heavy print slows down workers
            return torch.zeros(1), video_file, False

# --- 2. Configuration ---
video_transform = get_video_transform()

BATCH_SIZE = 1  # We yield 1 full video at a time
NUM_WORKERS = 2 # Start with 4. If unstable, reduce to 2.

# Setup Dataset
dataset = PreloadingVideoDataset(
    video_files,
    VIDEO_DIR,
    num_frames=NUM_FRAMES,
    segment_length=SEGMENT_LENGTH,
    transform=video_transform
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=1 # Buffer 2 videos per worker
)

# --- 3. Optimized Main Loop (The "GPU" Logic) ---
print(f"Extracting features using {NUM_WORKERS} workers with Adaptive Retry...")

failed_videos = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Processing Videos"):
        # Get data from worker (CPU work is already done!)
        video_tensors, filenames, valids = batch

        # DataLoader adds a batch dim (1, Segments, ...), remove it
        video_tensor = video_tensors[0]
        video_file = filenames[0]
        is_valid = valids[0]

        recording_id = os.path.splitext(video_file)[0]
        output_file = os.path.join(EGOVLP_FEATURE_DIR, f"{recording_id}.npz")

        if os.path.exists(output_file):
            continue

        if not is_valid:
            failed_videos.append(video_file)
            continue

        # --- RETRY LOGIC STARTS HERE ---
        # We try 3 different batch sizes: Large -> Medium -> Small
        possible_batch_sizes = [300, 128, 32]
        success = False

        for batch_size in possible_batch_sizes:
            try:
                # Clear memory before every attempt to give it the best chance
                torch.cuda.empty_cache()

                num_segments = len(video_tensor)
                num_batches = math.ceil(num_segments / batch_size)
                features_list = []

                with autocast():
                    for i in range(num_batches):
                        # Slice: Take segments i*batch_size to (i+1)*batch_size
                        chunk = video_tensor[i*batch_size : (i+1)*batch_size]
                        chunk = chunk.to(device) # Move to GPU

                        # model.compute_video expects (B, F, C, H, W)
                        batch_feat = model.compute_video(chunk)
                        features_list.append(batch_feat.cpu().numpy())

                final_features = np.vstack(features_list)
                np.savez(output_file, video_features=final_features)
                success = True
                break # Success! Exit the retry loop

            except RuntimeError as e:
                # Check if it is an OOM error
                if "out of memory" in str(e):
                    print(f"\n⚠️ OOM with batch {batch_size} on {video_file}. Retrying with smaller batch...")
                    torch.cuda.empty_cache()
                    continue # Try next size in the list
                else:
                    print(f"\n❌ Non-Memory Error on {video_file}: {e}")
                    break # Unknown error, don't retry
            except Exception as e:
                print(f"\n❌ Error on {video_file}: {e}")
                break

        if not success:
            print(f"Failed to process {video_file} after all attempts.")
            failed_videos.append(video_file)

        # Clean up System RAM
        del video_tensor
        gc.collect()

print(f"\nFeature extraction complete!")
print(f"Successful: {len(video_files) - len(failed_videos)}")
if failed_videos:
    print(f"Failed: {len(failed_videos)}")
    print(f"Failed videos: {failed_videos[:5]}...")

Extracting EgoVLP features for 384 videos...
Output directory: data/features/egovlp
Extracting features using 2 workers with Adaptive Retry...


Processing Videos:   0%|          | 0/384 [00:00<?, ?it/s]/tmp/ipython-input-3090502439.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Processing Videos: 100%|██████████| 384/384 [50:11<00:00,  7.84s/it]


Feature extraction complete!
Successful: 384


## 3. Feature Extraction with PerceptionEncoder (Alternative)

PerceptionEncoder is a newer video-text model. Setup instructions below.

**Note:** PerceptionEncoder may require different preprocessing. Check the official repository.

In [12]:
# PerceptionEncoder Setup
# Reference: https://arxiv.org/abs/2504.13181
# The PerceptionEncoder is available at: https://huggingface.co/facebook/perception-encoder

print("PerceptionEncoder can be loaded from HuggingFace:")
print("  from transformers import AutoModel")
print("  model = AutoModel.from_pretrained('facebook/perception-encoder')")
print()
print("Follow similar extraction pattern as EgoVLP.")
print("Key differences may include:")
print("  - Input resolution")
print("  - Number of frames")
print("  - Feature dimension")

PerceptionEncoder can be loaded from HuggingFace:
  from transformers import AutoModel
  model = AutoModel.from_pretrained('facebook/perception-encoder')

Follow similar extraction pattern as EgoVLP.
Key differences may include:
  - Input resolution
  - Number of frames
  - Feature dimension


## 4. Verify Extracted Features

Verify the extracted features are in the correct format and compare with existing features.

In [13]:
def verify_features(feature_dir, expected_dim=None):
    """
    Verify extracted features are in the correct format.

    Args:
        feature_dir: Directory containing .npz feature files
        expected_dim: Expected feature dimension (optional)
    """
    npz_files = sorted([f for f in os.listdir(feature_dir) if f.endswith('.npz')])
    print(f"Found {len(npz_files)} feature files in {feature_dir}")

    if not npz_files:
        print("No feature files found!")
        return

    # Sample statistics
    dims = []
    num_segments = []

    for npz_file in npz_files[:10]:  # Check first 10 files
        file_path = os.path.join(feature_dir, npz_file)
        data = np.load(file_path)

        # Try different key names
        if 'video_features' in data:
            features = data['video_features']
        elif 'arr_0' in data:
            features = data['arr_0']
        else:
            features = data[list(data.keys())[0]]

        dims.append(features.shape[-1])
        num_segments.append(features.shape[0])

    print(f"\nFeature Statistics (first {len(dims)} files):")
    print(f"  Feature dimension: {dims[0]} (consistent: {len(set(dims)) == 1})")
    print(f"  Segments range: {min(num_segments)} - {max(num_segments)}")

    if expected_dim and dims[0] != expected_dim:
        print(f"  WARNING: Expected dimension {expected_dim}, got {dims[0]}")

    # Show sample file details
    sample_file = os.path.join(feature_dir, npz_files[0])
    data = np.load(sample_file)
    print(f"\nSample file: {npz_files[0]}")
    print(f"  Keys: {list(data.keys())}")
    for key in data.keys():
        print(f"  {key}: shape={data[key].shape}, dtype={data[key].dtype}")


# Verify EgoVLP features
print("=" * 60)
print("EgoVLP Features")
print("=" * 60)
verify_features(EGOVLP_FEATURE_DIR, expected_dim=PROJECTION_DIM)

EgoVLP Features
Found 384 feature files in data/features/egovlp

Feature Statistics (first 10 files):
  Feature dimension: 256 (consistent: True)
  Segments range: 126 - 486

Sample file: 10_16_360p_224.npz
  Keys: ['video_features']
  video_features: shape=(486, 256), dtype=float16


## 5. Save Features to Google Drive

Copy extracted features to Google Drive for persistent storage.

In [14]:
# Copy EgoVLP features to Google Drive
import shutil

DRIVE_EGOVLP_DIR = f"{DRIVE_BASE_PATH}/features/egovlp"
os.makedirs(DRIVE_EGOVLP_DIR, exist_ok=True)

print(f"Copying EgoVLP features to {DRIVE_EGOVLP_DIR}...")
!cp -r {EGOVLP_FEATURE_DIR}/* "{DRIVE_EGOVLP_DIR}/"

# Verify copy
drive_files = os.listdir(DRIVE_EGOVLP_DIR)
print(f"Copied {len(drive_files)} files to Google Drive")

Copying EgoVLP features to /content/drive/MyDrive/AML_Project/features/egovlp...
Copied 384 files to Google Drive


## 6. Feature Comparison

Compare EgoVLP features with existing Omnivore/SlowFast features for the same recordings.

In [15]:
def compare_features(feature_dir1, feature_dir2, name1="Features1", name2="Features2"):
    """
    Compare features from two directories.
    """
    files1 = set([f for f in os.listdir(feature_dir1) if f.endswith('.npz')])
    files2 = set([f for f in os.listdir(feature_dir2) if f.endswith('.npz')])

    common_files = files1 & files2
    print(f"Common files: {len(common_files)}")
    print(f"{name1} only: {len(files1 - files2)}")
    print(f"{name2} only: {len(files2 - files1)}")

    if common_files:
        sample_file = list(common_files)[0]

        data1 = np.load(os.path.join(feature_dir1, sample_file))
        data2 = np.load(os.path.join(feature_dir2, sample_file))

        # Get features (handle different key names)
        f1 = data1['video_features'] if 'video_features' in data1 else data1[list(data1.keys())[0]]
        f2 = data2['video_features'] if 'video_features' in data2 else data2[list(data2.keys())[0]]

        print(f"\nSample: {sample_file}")
        print(f"  {name1}: {f1.shape}")
        print(f"  {name2}: {f2.shape}")

        # Segment count alignment
        seg_diff = abs(f1.shape[0] - f2.shape[0])
        if seg_diff > 0:
            print(f"  Segment count difference: {seg_diff}")


# If you have pre-extracted Omnivore features, uncomment:
# OMNIVORE_DIR = f"{DRIVE_BASE_PATH}/features/omnivore"  # Adjust path
# if os.path.exists(OMNIVORE_DIR):
#     print("Comparing EgoVLP with Omnivore features:")
#     compare_features(EGOVLP_FEATURE_DIR, OMNIVORE_DIR, "EgoVLP", "Omnivore")

print("Feature extraction complete!")
print(f"EgoVLP features saved to: {EGOVLP_FEATURE_DIR}")
print(f"Feature dimension: {PROJECTION_DIM}")
print(f"Segment length: {SEGMENT_LENGTH}s")

Feature extraction complete!
EgoVLP features saved to: data/features/egovlp
Feature dimension: 256
Segment length: 2s


## 7. Next Steps: Using EgoVLP Features for Task Verification

The extracted EgoVLP features can be used for the Task Verification extension:

1. **Load features**: Use `np.load(feature_path)['video_features']`
2. **Modify dataloader**: Update `CaptainCookStepDataset` to accept EgoVLP features
3. **Train classifier**: Use the same MLP/Transformer architecture with EgoVLP features
4. **Text alignment**: EgoVLP's video-text alignment enables step verification

```python
# Example: Using EgoVLP features with baseline model
from core.models.er_former import ERFormerModel

config = {
    'num_classes': 2,
    'input_dim': 256,  # EgoVLP projection dim
    'model_type': 'transformer',
    # ... other config
}

model = ERFormerModel(config)
# Load EgoVLP features instead of Omnivore
```